In [ ]:
# ============================================================
# Statistical analysis of paired per-image SR metrics
# ============================================================

from pathlib import Path
import sys
import platform

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.stats import (
    shapiro,
    friedmanchisquare,
    wilcoxon,
    rankdata,
    bootstrap
)

import statsmodels
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. Reproducibility settings
# ============================================================

RANDOM_SEED = 123
ALPHA = 0.05
N_BOOTSTRAP = 10_000
CONFIDENCE_LEVEL = 0.95

np.random.seed(RANDOM_SEED)


# ============================================================
# 2. Data location
# ============================================================

RESULTS_ROOT = Path(
    " path for results files for the proposed models and  SOTA(P-value, paired image)"
)

# RESULTS_ROOT = Path("Results")


# ============================================================
# 3. Model configuration
# ============================================================

MODEL_FILES = {
    "Proposed": RESULTS_ROOT / "proposed" / "metrics_per_image.csv",
    "ELAN":     RESULTS_ROOT / "ELAN" / "metrics_per_image.csv",
    "EDSR":     RESULTS_ROOT / "EDSR" / "metrics_per_image.csv",
    "SwinIR":   RESULTS_ROOT / "SwinIR" / "metrics_per_image.csv",
    "DAT":      RESULTS_ROOT / "DAT" / "metrics_per_image.csv",
    "GRL":      RESULTS_ROOT / "GRL" / "metrics_per_image.csv",
    "ESRGAN":   RESULTS_ROOT / "ESRGAN" / "metrics_per_image.csv",
}

MODEL_ORDER = [
    "Proposed",
    "ELAN",
    "EDSR",
    "SwinIR",
    "DAT",
    "GRL",
    "ESRGAN",
]

REFERENCE_MODELS = [
    "ELAN",
    "EDSR",
    "SwinIR",
    "DAT",
    "GRL",
    "ESRGAN",
]

METRICS = ["psnr", "ssim", "lpips", "dists"]

HIGHER_IS_BETTER = {
    "psnr": True,
    "ssim": True,
    "lpips": False,
    "dists": False,
}


# ============================================================
# 4. Environment information
# ============================================================

print("Python       :", sys.version.split()[0])
print("Platform     :", platform.platform())
print("NumPy        :", np.__version__)
print("pandas       :", pd.__version__)
print("SciPy        :", scipy.__version__)
print("statsmodels  :", statsmodels.__version__)

print("\nResults directory exists:", RESULTS_ROOT.exists())
print("Number of configured models:", len(MODEL_FILES))

In [ ]:
all_files_exist = True

for model_name, file_path in MODEL_FILES.items():
    exists = file_path.is_file()

    print(
        f"{model_name:10s} | "
        f"exists = {str(exists):5s} | "
        f"{file_path}"
    )

    if not exists:
        all_files_exist = False

if not all_files_exist:
    raise FileNotFoundError(
        "One or more metrics_per_image.csv files were not found."
    )

print("\nAll model files were found successfully.")

In [ ]:
# ============================================================
# 5. Read, clean, and validate each model file
# ============================================================

REQUIRED_COLUMNS = ["name", "psnr", "ssim", "lpips", "dists"]

models = {}
validation_rows = []

for model_name, file_path in MODEL_FILES.items():

    # --------------------------------------------------------
    # Read CSV
    # --------------------------------------------------------
    df_model = pd.read_csv(file_path)

    original_rows = len(df_model)

    # --------------------------------------------------------
    # Validate required columns
    # --------------------------------------------------------
    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in df_model.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{model_name}: missing required columns: "
            f"{missing_columns}"
        )

    # Keep only the required columns
    df_model = df_model[REQUIRED_COLUMNS].copy()

    # --------------------------------------------------------
    # Clean image names
    # --------------------------------------------------------
    df_model["name"] = (
        df_model["name"]
        .astype(str)
        .str.strip()
    )

    # Remove summary rows from the analysis copy only
    summary_mask = (
        df_model["name"]
        .str.upper()
        .isin(["MEAN", "STD"])
    )

    summary_rows_removed = int(summary_mask.sum())

    df_model = df_model.loc[~summary_mask].copy()

    # --------------------------------------------------------
    # Convert metric columns to numeric
    # --------------------------------------------------------
    for metric in METRICS:
        df_model[metric] = pd.to_numeric(
            df_model[metric],
            errors="coerce"
        )

    # --------------------------------------------------------
    # Data-quality checks
    # --------------------------------------------------------
    missing_values = int(
        df_model[REQUIRED_COLUMNS].isna().sum().sum()
    )

    duplicate_names = int(
        df_model["name"].duplicated().sum()
    )

    unique_images = int(
        df_model["name"].nunique()
    )

    # Check finite numerical values
    nonfinite_values = int(
        np.sum(
            ~np.isfinite(
                df_model[METRICS].to_numpy(dtype=float)
            )
        )
    )

    # Logical range checks
    invalid_psnr = int(
        (df_model["psnr"] <= 0).sum()
    )

    invalid_ssim = int(
        (
            (df_model["ssim"] < 0) |
            (df_model["ssim"] > 1)
        ).sum()
    )

    invalid_lpips = int(
        (df_model["lpips"] < 0).sum()
    )

    invalid_dists = int(
        (df_model["dists"] < 0).sum()
    )

    # --------------------------------------------------------
    # Stop if a critical problem is found
    # --------------------------------------------------------
    if missing_values > 0:
        raise ValueError(
            f"{model_name}: found {missing_values} missing values."
        )

    if nonfinite_values > 0:
        raise ValueError(
            f"{model_name}: found {nonfinite_values} "
            "non-finite metric values."
        )

    if duplicate_names > 0:
        raise ValueError(
            f"{model_name}: found {duplicate_names} "
            "duplicate image names."
        )

    if invalid_psnr > 0:
        raise ValueError(
            f"{model_name}: found invalid PSNR values."
        )

    if invalid_ssim > 0:
        raise ValueError(
            f"{model_name}: found SSIM values outside [0, 1]."
        )

    if invalid_lpips > 0:
        raise ValueError(
            f"{model_name}: found negative LPIPS values."
        )

    if invalid_dists > 0:
        raise ValueError(
            f"{model_name}: found negative DISTS values."
        )

    # --------------------------------------------------------
    # Sort by image name for reproducible output
    # --------------------------------------------------------
    df_model = (
        df_model
        .sort_values("name")
        .reset_index(drop=True)
    )

    models[model_name] = df_model

    validation_rows.append({
        "Model": model_name,
        "Original_Rows": original_rows,
        "Summary_Rows_Removed": summary_rows_removed,
        "Images_After_Cleaning": len(df_model),
        "Unique_Images": unique_images,
        "Missing_Values": missing_values,
        "Duplicate_Names": duplicate_names,
        "Nonfinite_Values": nonfinite_values,
        "PSNR_Min": df_model["psnr"].min(),
        "PSNR_Max": df_model["psnr"].max(),
        "SSIM_Min": df_model["ssim"].min(),
        "SSIM_Max": df_model["ssim"].max(),
        "LPIPS_Min": df_model["lpips"].min(),
        "LPIPS_Max": df_model["lpips"].max(),
        "DISTS_Min": df_model["dists"].min(),
        "DISTS_Max": df_model["dists"].max(),
    })


validation_df = pd.DataFrame(validation_rows)

display(
    validation_df.style.format({
        "PSNR_Min": "{:.6f}",
        "PSNR_Max": "{:.6f}",
        "SSIM_Min": "{:.6f}",
        "SSIM_Max": "{:.6f}",
        "LPIPS_Min": "{:.6f}",
        "LPIPS_Max": "{:.6f}",
        "DISTS_Min": "{:.6f}",
        "DISTS_Max": "{:.6f}",
    })
)

print("\nAll model files passed the validation checks.")

In [ ]:
# ============================================================
# 6. Match image IDs across all models
#    and build the master paired dataset
# ============================================================

OUTPUT_DIR = RESULTS_ROOT / "statistical_analysis_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

reference_model = "Proposed"
reference_names = set(models[reference_model]["name"])

matching_report = []

for model_name in MODEL_ORDER:

    current_names = set(models[model_name]["name"])

    missing_from_model = reference_names - current_names
    extra_in_model = current_names - reference_names

    matching_report.append({
        "Model": model_name,
        "Number_of_Images": len(models[model_name]),
        "Missing_vs_Proposed": len(missing_from_model),
        "Extra_vs_Proposed": len(extra_in_model),
        "Exact_Name_Match": (
            len(missing_from_model) == 0
            and len(extra_in_model) == 0
        )
    })

matching_report_df = pd.DataFrame(matching_report)

display(matching_report_df)

if not matching_report_df["Exact_Name_Match"].all():
    raise ValueError(
        "Image-name mismatch detected between the models."
    )

print("\nAll models contain exactly the same image IDs.")

In [ ]:
# ============================================================
# 7. Build one-to-one paired master table
# ============================================================

merged_df = models["Proposed"][["name"]].copy()

for model_name in MODEL_ORDER:

    current_df = models[model_name][
        ["name"] + METRICS
    ].copy()

    current_df = current_df.rename(
        columns={
            metric: f"{model_name}_{metric}"
            for metric in METRICS
        }
    )

    merged_df = merged_df.merge(
        current_df,
        on="name",
        how="inner",
        validate="one_to_one"
    )

merged_df = (
    merged_df
    .sort_values("name")
    .reset_index(drop=True)
)

expected_columns = 1 + len(MODEL_ORDER) * len(METRICS)

print("Master dataset shape:", merged_df.shape)
print("Expected columns    :", expected_columns)
print("Missing values      :", merged_df.isna().sum().sum())
print("Duplicated names    :", merged_df["name"].duplicated().sum())

if len(merged_df) != len(reference_names):
    raise ValueError(
        "The merged dataset does not contain all paired images."
    )

if merged_df.shape[1] != expected_columns:
    raise ValueError(
        "Unexpected number of columns in the merged dataset."
    )

if merged_df.isna().sum().sum() > 0:
    raise ValueError(
        "Missing values were introduced during merging."
    )

if merged_df["name"].duplicated().any():
    raise ValueError(
        "Duplicate image names were introduced during merging."
    )

print("\nMaster paired dataset created successfully.")

In [ ]:
# ============================================================
# 8. Save validated intermediate outputs
# ============================================================

master_dataset_path = (
    OUTPUT_DIR / "master_paired_metrics.csv"
)

matching_report_path = (
    OUTPUT_DIR / "image_matching_report.csv"
)

validation_report_path = (
    OUTPUT_DIR / "data_validation_report.csv"
)

merged_df.to_csv(
    master_dataset_path,
    index=False
)

matching_report_df.to_csv(
    matching_report_path,
    index=False
)

validation_df.to_csv(
    validation_report_path,
    index=False
)

print("Saved:")
print(master_dataset_path)
print(matching_report_path)
print(validation_report_path)

In [ ]:
# ============================================================
# 9. Descriptive statistics for all models and metrics
# ============================================================

descriptive_rows = []

for model_name in MODEL_ORDER:

    for metric in METRICS:

        values = models[model_name][metric].to_numpy(dtype=float)

        q1 = np.percentile(values, 25)
        q3 = np.percentile(values, 75)

        descriptive_rows.append({
            "Model": model_name,
            "Metric": metric.upper(),
            "N": len(values),
            "Mean": np.mean(values),
            "SD": np.std(values, ddof=1),
            "Median": np.median(values),
            "Q1": q1,
            "Q3": q3,
            "IQR": q3 - q1,
            "Minimum": np.min(values),
            "Maximum": np.max(values),
        })

descriptive_df = pd.DataFrame(descriptive_rows)

descriptive_df["Metric"] = pd.Categorical(
    descriptive_df["Metric"],
    categories=["PSNR", "SSIM", "LPIPS", "DISTS"],
    ordered=True
)

descriptive_df["Model"] = pd.Categorical(
    descriptive_df["Model"],
    categories=MODEL_ORDER,
    ordered=True
)

descriptive_df = (
    descriptive_df
    .sort_values(["Metric", "Model"])
    .reset_index(drop=True)
)

display(
    descriptive_df.style.format({
        "Mean": "{:.6f}",
        "SD": "{:.6f}",
        "Median": "{:.6f}",
        "Q1": "{:.6f}",
        "Q3": "{:.6f}",
        "IQR": "{:.6f}",
        "Minimum": "{:.6f}",
        "Maximum": "{:.6f}",
    })
)

In [ ]:
# ============================================================
# 10. Compact descriptive table for publication
# ============================================================

def format_mean_sd(row):
    metric = str(row["Metric"])

    if metric == "PSNR":
        return f"{row['Mean']:.3f} ± {row['SD']:.3f}"
    else:
        return f"{row['Mean']:.5f} ± {row['SD']:.5f}"


def format_median_iqr(row):
    metric = str(row["Metric"])

    if metric == "PSNR":
        return (
            f"{row['Median']:.3f} "
            f"[{row['Q1']:.3f}, {row['Q3']:.3f}]"
        )
    else:
        return (
            f"{row['Median']:.5f} "
            f"[{row['Q1']:.5f}, {row['Q3']:.5f}]"
        )


descriptive_publish = descriptive_df.copy()

descriptive_publish["Mean ± SD"] = (
    descriptive_publish.apply(format_mean_sd, axis=1)
)

descriptive_publish["Median [Q1, Q3]"] = (
    descriptive_publish.apply(format_median_iqr, axis=1)
)

descriptive_publish = descriptive_publish[
    [
        "Metric",
        "Model",
        "N",
        "Mean ± SD",
        "Median [Q1, Q3]"
    ]
]

display(descriptive_publish)

In [ ]:
descriptive_full_path = (
    OUTPUT_DIR / "descriptive_statistics_full.csv"
)

descriptive_publish_path = (
    OUTPUT_DIR / "descriptive_statistics_publication.csv"
)

descriptive_df.to_csv(
    descriptive_full_path,
    index=False
)

descriptive_publish.to_csv(
    descriptive_publish_path,
    index=False
)

print("Saved:")
print(descriptive_full_path)
print(descriptive_publish_path)

In [ ]:
# ============================================================
# Global comparison using the Friedman test
# ============================================================

from scipy.stats import friedmanchisquare

friedman_rows = []

n_images = len(merged_df)
n_models = len(MODEL_ORDER)

for metric in METRICS:

    metric_arrays = [
        merged_df[f"{model}_{metric}"].to_numpy(dtype=float)
        for model in MODEL_ORDER
    ]

    chi_square, p_value = friedmanchisquare(*metric_arrays)

    kendalls_w = chi_square / (
        n_images * (n_models - 1)
    )

    friedman_rows.append({
        "Metric": metric.upper(),
        "Friedman χ²": chi_square,
        "p-value": p_value,
        "Kendall's W": kendalls_w
    })


table2_full = pd.DataFrame(friedman_rows)

table2_full["Metric"] = pd.Categorical(
    table2_full["Metric"],
    categories=["PSNR", "SSIM", "LPIPS", "DISTS"],
    ordered=True
)

table2_full = (
    table2_full
    .sort_values("Metric")
    .reset_index(drop=True)
)


# Publication-ready copy
table2_publish = table2_full.copy()

table2_publish["p-value"] = (
    table2_publish["p-value"].apply(
        lambda value: (
            "< 0.001"
            if value < 0.001
            else f"{value:.3f}"
        )
    )
)

table2_publish["Friedman χ²"] = (
    table2_publish["Friedman χ²"].map(
        lambda value: f"{value:.3f}"
    )
)

table2_publish["Kendall's W"] = (
    table2_publish["Kendall's W"].map(
        lambda value: f"{value:.3f}"
    )
)

display(table2_publish)

In [ ]:
table2_full_path = (
    OUTPUT_DIR / "table2_friedman_full.csv"
)

table2_publish_path = (
    OUTPUT_DIR / "table2_friedman_publication.csv"
)

table2_full.to_csv(
    table2_full_path,
    index=False
)

table2_publish.to_csv(
    table2_publish_path,
    index=False
)

print("Saved:")
print(table2_full_path)
print(table2_publish_path)

In [ ]:
#-Wilcoxon: Metric، Reference Model، Median Difference (95% CI)، وHolm-adjusted p-value
# ============================================================
# ============================================================

from scipy.stats import wilcoxon, bootstrap
from statsmodels.stats.multitest import multipletests


def paired_difference(
    proposed_values,
    reference_values
):
    """
    Return paired differences defined as:

        Proposed - Reference

    for all metrics.
    """

    proposed_values = np.asarray(
        proposed_values,
        dtype=float
    )

    reference_values = np.asarray(
        reference_values,
        dtype=float
    )

    valid_mask = (
        np.isfinite(proposed_values)
        & np.isfinite(reference_values)
    )

    return (
        proposed_values[valid_mask]
        - reference_values[valid_mask]
    )


def bootstrap_median_ci(
    differences,
    confidence_level=0.95,
    n_resamples=10_000,
    random_seed=123
):
    """
    BCa bootstrap confidence interval for the median
    of paired differences.
    """

    differences = np.asarray(
        differences,
        dtype=float
    )

    result = bootstrap(
        data=(differences,),
        statistic=np.median,
        confidence_level=confidence_level,
        n_resamples=n_resamples,
        method="BCa",
        random_state=random_seed
    )

    return (
        float(result.confidence_interval.low),
        float(result.confidence_interval.high)
    )

In [ ]:
# ============================================================
# Wilcoxon post-hoc paired comparisons
# ============================================================

pairwise_rows = []

for metric in METRICS:

    for reference_model in REFERENCE_MODELS:

        proposed_values = merged_df[
            f"Proposed_{metric}"
        ].to_numpy(dtype=float)

        reference_values = merged_df[
            f"{reference_model}_{metric}"
        ].to_numpy(dtype=float)

        differences = paired_difference(
            proposed_values,
            reference_values
        )

        median_difference = float(
            np.median(differences)
        )

        ci_low, ci_high = bootstrap_median_ci(
            differences=differences,
            confidence_level=CONFIDENCE_LEVEL,
            n_resamples=N_BOOTSTRAP,
            random_seed=RANDOM_SEED
        )

        wilcoxon_statistic, raw_p_value = wilcoxon(
            differences,
            zero_method="wilcox",
            correction=False,
            alternative="two-sided",
            mode="approx"
        )

        pairwise_rows.append({
            "Metric": metric.upper(),
            "Reference Model": reference_model,
            "N": len(differences),
            "Median Difference": median_difference,
            "CI95 Lower": ci_low,
            "CI95 Upper": ci_high,
            "Wilcoxon Statistic": wilcoxon_statistic,
            "Raw p-value": raw_p_value
        })


table3_full = pd.DataFrame(pairwise_rows)

In [ ]:
# ============================================================
# Holm correction within each metric
# ============================================================

table3_full["Holm-adjusted p-value"] = np.nan

for metric_name in [
    "PSNR",
    "SSIM",
    "LPIPS",
    "DISTS"
]:

    metric_mask = (
        table3_full["Metric"] == metric_name
    )

    raw_p_values = table3_full.loc[
        metric_mask,
        "Raw p-value"
    ].to_numpy()

    _, adjusted_p_values, _, _ = multipletests(
        raw_p_values,
        alpha=ALPHA,
        method="holm"
    )

    table3_full.loc[
        metric_mask,
        "Holm-adjusted p-value"
    ] = adjusted_p_values

In [ ]:
table3_full["Metric"] = pd.Categorical(
    table3_full["Metric"],
    categories=["PSNR", "SSIM", "LPIPS", "DISTS"],
    ordered=True
)

reference_order = [
    "DAT",
    "EDSR",
    "ELAN",
    "ESRGAN",
    "GRL",
    "SwinIR"
]

table3_full["Reference Model"] = pd.Categorical(
    table3_full["Reference Model"],
    categories=reference_order,
    ordered=True
)

table3_full = (
    table3_full
    .sort_values(
        ["Metric", "Reference Model"]
    )
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# Publication-ready Table 3
# ============================================================

def format_difference_ci(row):

    metric = str(row["Metric"])

    median_difference = row["Median Difference"]
    ci_low = row["CI95 Lower"]
    ci_high = row["CI95 Upper"]

    if metric == "PSNR":
        return (
            f"{median_difference:.3f} "
            f"[{ci_low:.3f}, {ci_high:.3f}]"
        )

    return (
        f"{median_difference:.5f} "
        f"[{ci_low:.5f}, {ci_high:.5f}]"
    )


def format_adjusted_p(value):

    if value < 0.001:
        return "< 0.001"

    return f"{value:.3f}"


table3_publish = table3_full.copy()

table3_publish[
    "Median Difference (95% CI)"
] = table3_publish.apply(
    format_difference_ci,
    axis=1
)

table3_publish[
    "Holm-adjusted p-value"
] = table3_publish[
    "Holm-adjusted p-value"
].apply(format_adjusted_p)

table3_publish = table3_publish[
    [
        "Metric",
        "Reference Model",
        "Median Difference (95% CI)",
        "Holm-adjusted p-value"
    ]
]

display(table3_publish)

In [ ]:
table3_full_path = (
    OUTPUT_DIR / "table3_wilcoxon_full.csv"
)

table3_publish_path = (
    OUTPUT_DIR / "table3_wilcoxon_publication.csv"
)

table3_full.to_csv(
    table3_full_path,
    index=False
)

table3_publish.to_csv(
    table3_publish_path,
    index=False
)

print("Saved:")
print(table3_full_path)
print(table3_publish_path)

In [ ]:
#--------------- PLOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1. PSNR paired median differences and BCa 95% CIs
#    Difference is defined as Proposed - Reference
# ============================================================

psnr_forest_df = pd.DataFrame({
    "Model": [
        "DAT",
        "GRL",
        "EDSR",
        "ESRGAN",
        "SwinIR",
        "ELAN"
    ],
    "Median": [
        0.519,
        0.419,
        1.243,
        2.393,
        3.362,
        4.015
    ],
    "Lower": [
        0.504,
        0.411,
        1.222,
        2.372,
        3.306,
        3.964
    ],
    "Upper": [
        0.530,
        0.432,
        1.269,
        2.413,
        3.400,
        4.063
    ]
})

psnr_forest_df["Minus_Error"] = (
    psnr_forest_df["Median"] -
    psnr_forest_df["Lower"]
)

psnr_forest_df["Plus_Error"] = (
    psnr_forest_df["Upper"] -
    psnr_forest_df["Median"]
)

display(psnr_forest_df)

In [ ]:
# ============================================================
# 2. Draw the PSNR forest plot
# ============================================================

plot_df = psnr_forest_df.iloc[::-1].reset_index(drop=True)

y_positions = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(6, 3.8))

ax.errorbar(
    x=plot_df["Median"],
    y=y_positions,
    xerr=np.vstack([
        plot_df["Minus_Error"],
        plot_df["Plus_Error"]
    ]),
    fmt="o",
    markersize=7,
    capsize=3,
    linewidth=1.2,
    elinewidth=1
)

ax.axvline(
    x=0,
    linestyle="--",
    linewidth=1
)

ax.set_yticks(y_positions)
ax.set_yticklabels(plot_df["Model"])

ax.set_xlabel(
    "Median paired difference in PSNR (Proposed − Reference), dB"
)

ax.set_ylabel("Models")

ax.set_title(
    "Paired Median Differences in PSNR"
)

ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.7
)

x_min = min(0, plot_df["Lower"].min())
x_max = plot_df["Upper"].max()

margin = 0.05 * (x_max - x_min)

ax.set_xlim(
    x_min - margin,
    x_max + margin
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Draw the publication-ready PSNR forest plot
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Serif"

model_order = [
    "ELAN",
    "SwinIR",
    "ESRGAN",
    "EDSR",
    "DAT",
    "GRL"
]

plot_df = (
    psnr_forest_df
    .set_index("Model")
    .loc[model_order]
    .reset_index()
)

plot_df = plot_df.iloc[::-1].reset_index(drop=True)

y_positions = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(6, 3.8))

ax.errorbar(
    x=plot_df["Median"],
    y=y_positions,
    xerr=np.vstack([
        plot_df["Minus_Error"],
        plot_df["Plus_Error"]
    ]),
    fmt="o",
    color="black",
    ecolor="black",
    markerfacecolor="black",
    markeredgecolor="black",
    markersize=7,
    capsize=3,
    linewidth=1,
    elinewidth=1
)

ax.axvline(
    x=0,
    linestyle="--",
    linewidth=0.8,
    color="gray"
)

ax.set_yticks(y_positions)
ax.set_yticklabels(
    plot_df["Model"],
    fontsize=10
)

ax.set_xlabel(
    "Median paired difference in PSNR "
    "(Proposed − Reference), dB",
    fontsize=11
)

ax.set_ylabel(
    "Compared model",
    fontsize=11
)

ax.set_title("")

ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.6,
    color="0.75"
)

ax.tick_params(
    axis="y",
    pad=8,
    labelsize=10
)

ax.tick_params(
    axis="x",
    labelsize=10
)

x_min = 0
x_max = plot_df["Upper"].max()

margin = 0.04 * (x_max - x_min)

ax.set_xlim(
    x_min - margin,
    x_max + margin
)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)

plt.tight_layout()
plt.show()

In [ ]:
output_dir = Path("write output path")

output_dir.mkdir(parents=True, exist_ok=True)

png_path = output_dir / "forest_plot_psnr.png"
pdf_path = output_dir / "forest_plot_psnr.pdf"
svg_path = output_dir / "forest_plot_psnr.svg"

plt.tight_layout()

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

fig.savefig(
    svg_path,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(png_path)
print(pdf_path)
print(svg_path)

In [ ]:
# ssim--------
# ============================================================
# SSIM paired median differences and BCa 95% CIs
# Difference is defined as Proposed - Reference
# ============================================================

ssim_forest_df = pd.DataFrame({
    "Model": [
        "DAT",
        "EDSR",
        "ELAN",
        "ESRGAN",
        "GRL",
        "SwinIR"
    ],
    "Median": [
        0.00724,
        0.01335,
        0.04611,
        0.04797,
        0.00508,
        0.03286
    ],
    "Lower": [
        0.00719,
        0.01324,
        0.04583,
        0.04779,
        0.00499,
        0.03259
    ],
    "Upper": [
        0.00732,
        0.01346,
        0.04638,
        0.04830,
        0.00516,
        0.03315
    ]
})

ssim_forest_df["Minus_Error"] = (
    ssim_forest_df["Median"] -
    ssim_forest_df["Lower"]
)

ssim_forest_df["Plus_Error"] = (
    ssim_forest_df["Upper"] -
    ssim_forest_df["Median"]
)

display(ssim_forest_df)

In [ ]:
# ============================================================
# Draw the publication-ready SSIM forest plot
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Serif"

model_order = [
    "ELAN",
    "SwinIR",
    "ESRGAN",
    "EDSR",
    "DAT",
    "GRL"
]

plot_df = (
    ssim_forest_df
    .set_index("Model")
    .loc[model_order]
    .reset_index()
)

plot_df = plot_df.iloc[::-1].reset_index(drop=True)

y_positions = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(6, 3.8))

ax.errorbar(
    x=plot_df["Median"],
    y=y_positions,
    xerr=np.vstack([
        plot_df["Minus_Error"],
        plot_df["Plus_Error"]
    ]),
    fmt="o",
    color="black",
    ecolor="black",
    markerfacecolor="black",
    markeredgecolor="black",
    markersize=7,
    capsize=3,
    linewidth=1,
    elinewidth=1
)

ax.axvline(
    x=0,
    linestyle="--",
    linewidth=0.8,
    color="gray"
)

ax.set_yticks(y_positions)
ax.set_yticklabels(
    plot_df["Model"],
    fontsize=10
)

ax.set_xlabel(
    "Median paired difference in SSIM "
    "(Proposed − Reference)",
    fontsize=11
)

ax.set_ylabel(
    "Compared model",
    fontsize=11
)

ax.set_title("")

ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.6,
    color="0.75"
)

ax.tick_params(
    axis="y",
    pad=8,
    labelsize=10
)

ax.tick_params(
    axis="x",
    labelsize=10
)

x_min = 0
x_max = plot_df["Upper"].max()

margin = 0.04 * (x_max - x_min)

ax.set_xlim(
    x_min - margin,
    x_max + margin
)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Save publication-quality SSIM forest plot
# ============================================================

from pathlib import Path

output_dir = Path("output path")

output_dir.mkdir(parents=True, exist_ok=True)

png_file = output_dir / "Figure_SSIM_ForestPlot.png"
pdf_file = output_dir / "Figure_SSIM_ForestPlot.pdf"
svg_file = output_dir / "Figure_SSIM_ForestPlot.svg"

plt.tight_layout()

fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved successfully:")
print(png_file)
print(pdf_file)
print(svg_file)

In [ ]:
# lpips
# ============================================================
# LPIPS paired median differences and BCa 95% CIs
# Difference is defined as Proposed - Reference
# ============================================================

lpips_forest_df = pd.DataFrame({
    "Model": [
        "DAT",
        "EDSR",
        "ELAN",
        "ESRGAN",
        "GRL",
        "SwinIR"
    ],
    "Median": [
        -0.00889,
        -0.01084,
        -0.03784,
         0.01918,
        -0.00725,
        -0.03162
    ],
    "Lower": [
        -0.00903,
        -0.01105,
        -0.03886,
         0.01850,
        -0.00736,
        -0.03210
    ],
    "Upper": [
        -0.00875,
        -0.01067,
        -0.03695,
         0.01962,
        -0.00709,
        -0.03126
    ]
})

lpips_forest_df["Minus_Error"] = (
    lpips_forest_df["Median"] -
    lpips_forest_df["Lower"]
)

lpips_forest_df["Plus_Error"] = (
    lpips_forest_df["Upper"] -
    lpips_forest_df["Median"]
)

display(lpips_forest_df)

In [ ]:
# ============================================================
# Draw the publication-ready LPIPS forest plot
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Serif"

model_order = [
    "ELAN",
    "SwinIR",
    "ESRGAN",
    "EDSR",
    "DAT",
    "GRL"
]

plot_df = (
    lpips_forest_df
    .set_index("Model")
    .loc[model_order]
    .reset_index()
)

plot_df = plot_df.iloc[::-1].reset_index(drop=True)

y_positions = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(6, 3.8))

ax.errorbar(
    x=plot_df["Median"],
    y=y_positions,
    xerr=np.vstack([
        plot_df["Minus_Error"],
        plot_df["Plus_Error"]
    ]),
    fmt="o",
    color="black",
    ecolor="black",
    markerfacecolor="black",
    markeredgecolor="black",
    markersize=7,
    capsize=3,
    linewidth=1,
    elinewidth=1
)

ax.axvline(
    x=0,
    linestyle="--",
    linewidth=0.8,
    color="gray"
)

ax.set_yticks(y_positions)
ax.set_yticklabels(
    plot_df["Model"],
    fontsize=10
)

ax.set_xlabel(
    "Median paired difference in LPIPS "
    "(Proposed − Reference)",
    fontsize=11
)

ax.set_ylabel(
    "Compared model",
    fontsize=11
)

ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.6,
    color="0.75"
)

ax.tick_params(
    axis="y",
    pad=8,
    labelsize=10
)

ax.tick_params(
    axis="x",
    labelsize=10
)

x_min = plot_df["Lower"].min()
x_max = plot_df["Upper"].max()

margin = 0.05 * (x_max - x_min)

ax.set_xlim(
    x_min - margin,
    x_max + margin
)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Save publication-quality LPIPS forest plot
# ============================================================

from pathlib import Path

output_dir = Path(
    "/home/sudad/HiFi_project/1-Mamba project using new data/"
    "-Final Ablation for 30 epoch on lr-g=2.e4/"
    "analysis of MAX fusion with SOTA(P-value, paired image)/"
    "Results/statistical_analysis_outputs"
)

output_dir.mkdir(parents=True, exist_ok=True)

png_file = output_dir / "Figure_LPIPS_ForestPlot.png"
pdf_file = output_dir / "Figure_LPIPS_ForestPlot.pdf"
svg_file = output_dir / "Figure_LPIPS_ForestPlot.svg"

plt.tight_layout()

fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved successfully:")
print(png_file)
print(pdf_file)
print(svg_file)

In [ ]:
# ============================================================
# DISTS paired median differences and BCa 95% CIs
# Difference is defined as Proposed - Reference
# ============================================================

dists_forest_df = pd.DataFrame({
    "Model": [
        "DAT",
        "EDSR",
        "ELAN",
        "ESRGAN",
        "GRL",
        "SwinIR"
    ],
    "Median": [
        -0.00719,
        -0.01153,
        -0.05134,
         0.02807,
        -0.00395,
        -0.04827
    ],
    "Lower": [
        -0.00762,
        -0.01202,
        -0.05218,
         0.02768,
        -0.00430,
        -0.04921
    ],
    "Upper": [
        -0.00688,
        -0.01107,
        -0.05045,
         0.02854,
        -0.00366,
        -0.04758
    ]
})

dists_forest_df["Minus_Error"] = (
    dists_forest_df["Median"] -
    dists_forest_df["Lower"]
)

dists_forest_df["Plus_Error"] = (
    dists_forest_df["Upper"] -
    dists_forest_df["Median"]
)

display(dists_forest_df)

In [ ]:
# ============================================================
# Draw the publication-ready DISTS forest plot
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Serif"

model_order = [
    "ELAN",
    "SwinIR",
    "ESRGAN",
    "EDSR",
    "DAT",
    "GRL"
]

plot_df = (
    dists_forest_df
    .set_index("Model")
    .loc[model_order]
    .reset_index()
)

plot_df = plot_df.iloc[::-1].reset_index(drop=True)

y_positions = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(6, 3.8))

ax.errorbar(
    x=plot_df["Median"],
    y=y_positions,
    xerr=np.vstack([
        plot_df["Minus_Error"],
        plot_df["Plus_Error"]
    ]),
    fmt="o",
    color="black",
    ecolor="black",
    markerfacecolor="black",
    markeredgecolor="black",
    markersize=7,
    capsize=3,
    linewidth=1,
    elinewidth=1
)

# خط عدم وجود فرق
ax.axvline(
    x=0,
    linestyle="--",
    linewidth=0.8,
    color="gray"
)

# أسماء النماذج
ax.set_yticks(y_positions)
ax.set_yticklabels(
    plot_df["Model"],
    fontsize=10
)

# عناوين المحاور
ax.set_xlabel(
    "Median paired difference in DISTS "
    "(Proposed − Reference)",
    fontsize=11
)

ax.set_ylabel(
    "Compared model",
    fontsize=11
)

# شبكة أفقية خفيفة
ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.6,
    color="0.75"
)

ax.tick_params(
    axis="y",
    pad=8,
    labelsize=10
)

ax.tick_params(
    axis="x",
    labelsize=10
)

# حدود المحور
x_min = plot_df["Lower"].min()
x_max = plot_df["Upper"].max()

margin = 0.05 * (x_max - x_min)

ax.set_xlim(
    x_min - margin,
    x_max + margin
)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Save publication-quality DISTS forest plot
# ============================================================

from pathlib import Path

output_dir = Path(
    "/home/sudad/HiFi_project/1-Mamba project using new data/"
    "-Final Ablation for 30 epoch on lr-g=2.e4/"
    "analysis of MAX fusion with SOTA(P-value, paired image)/"
    "Results/statistical_analysis_outputs"
)

output_dir.mkdir(parents=True, exist_ok=True)

png_file = output_dir / "Figure_DISTS_ForestPlot.png"
pdf_file = output_dir / "Figure_DISTS_ForestPlot.pdf"
svg_file = output_dir / "Figure_DISTS_ForestPlot.svg"

plt.tight_layout()

fig.savefig(
    png_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    svg_file,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved successfully:")
print(png_file)
print(pdf_file)
print(svg_file)